In [1]:
import psycopg2
import os
import pandas as pd

# Kết nối (như cũ)
conn = psycopg2.connect(
    host=os.environ.get('DB_HOSTNAME'),
    user=os.environ.get('DB_USERNAME'),
    password=os.environ.get('DB_PASSWORD'),
    dbname=os.environ.get('DB_DATABASE')
)

# Đếm số lượng ảnh theo từng loại sản phẩm
sql = """
SELECT t.name as product_name, count(*) as total_images
FROM agdc.dataset d
JOIN agdc.dataset_type t ON d.dataset_type_ref = t.id
GROUP BY t.name
ORDER BY total_images DESC;
"""

df = pd.read_sql(sql, conn)
print(df)
conn.close()

/tmp/ipykernel_218/1243805045.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


                                product_name  total_images
0                                     s2_l2a       2074896
1                          sentinel_2_c1_l2a       1137446
2                           landsat7_c2l2_sr        169065
3                           landsat7_c2l2_st        168410
4                           landsat8_c2l2_sr        122846
5                           landsat8_c2l2_st        120778
6                           landsat5_c2l2_sr        118583
7                           landsat5_c2l2_st        118406
8                              landsat8_c2l1        111513
9                            nasa_aqua_l2_oc         37232
10                          landsat9_c2l2_sr         30214
11                          landsat9_c2l2_st         29709
12                             landsat9_c2l1         18875
13                          nasa_aqua_l2_sst         12070
14              ga_ls_mangrove_cover_cyear_3          5580
15         sentinel1_grd_gamma0_10m_unsmooth          54

In [1]:
import psycopg2
import os

# 1. Lấy thông tin kết nối
db_host = os.environ.get('DB_HOSTNAME')
db_user = os.environ.get('DB_USERNAME')
db_pass = os.environ.get('DB_PASSWORD')
db_name = os.environ.get('DB_DATABASE')

# 2. Danh sách các bảng cần lấy
tables = [
    'agdc.metadata_type',
    'agdc.dataset_type',
    'agdc.dataset_location',
    'agdc.dataset'
]

print("Đang kết nối trực tiếp tới PostgreSQL...")

try:
    # Kết nối trực tiếp (Bỏ qua Pandas/SQLAlchemy)
    conn = psycopg2.connect(
        host=db_host,
        user=db_user,
        password=db_pass,
        dbname=db_name
    )
    cur = conn.cursor()
    print("Kết nối thành công!\n")

    for table_name in tables:
        print(f"--> Đang xuất bảng: {table_name}")
        
        # Tên file CSV đầu ra
        file_name = table_name.split('.')[1] + ".csv"
        
        # Sử dụng lệnh COPY chuyên dụng của Postgres (Cực nhanh và chuẩn)
        # SQL: COPY (SELECT * FROM table) TO STDOUT WITH CSV HEADER
        sql = f"COPY (SELECT * FROM {table_name}) TO STDOUT WITH CSV HEADER"
        
        with open(file_name, 'w') as f:
            cur.copy_expert(sql, f)
            
        print(f"    Đã lưu xong: {file_name}")

    cur.close()
    conn.close()
    print("\nHOÀN TẤT! Bạn hãy tải 4 file CSV về máy.")

except Exception as e:
    print(f"\nCÓ LỖI XẢY RA: {e}")

Đang kết nối trực tiếp tới PostgreSQL...
Kết nối thành công!

--> Đang xuất bảng: agdc.metadata_type
    Đã lưu xong: metadata_type.csv
--> Đang xuất bảng: agdc.dataset_type
    Đã lưu xong: dataset_type.csv
--> Đang xuất bảng: agdc.dataset_location
    Đã lưu xong: dataset_location.csv
--> Đang xuất bảng: agdc.dataset
    Đã lưu xong: dataset.csv

HOÀN TẤT! Bạn hãy tải 4 file CSV về máy.


In [ ]:
import psycopg2
import os
import re # Thư viện xử lý chuỗi để lấy số liệu

# --- CẤU HÌNH ---
YEARS = [2020, 2021, 2022, 2023, 2024]
AVG_ROW_SIZE_KB = 4 

db_host = os.environ.get('DB_HOSTNAME')
db_user = os.environ.get('DB_USERNAME')
db_pass = os.environ.get('DB_PASSWORD')
db_name = os.environ.get('DB_DATABASE')

print("Đang kết nối...")

try:
    conn = psycopg2.connect(
        host=db_host, user=db_user, password=db_pass, dbname=db_name
    )
    cur = conn.cursor()
    print("Kết nối thành công! Đang ước lượng nhanh...\n")

    # ==========================================
    # BƯỚC 1: ƯỚC LƯỢNG SIÊU TỐC (INSTANT ESTIMATE)
    # ==========================================
    print("--- BẢNG DỰ TÍNH DUNG LƯỢNG (ƯỚC LƯỢNG) ---")
    print(f"{'NĂM':<10} | {'SỐ ẢNH (ƯỚC TÍNH)':<20} | {'SIZE (CSV)':<20}")
    print("-" * 60)

    total_est_rows = 0

    for year in YEARS:
        # Mẹo: Dùng EXPLAIN để lấy số liệu ước tính từ Query Planner
        # Nó sẽ trả về chuỗi kiểu: "Seq Scan on dataset ... (rows=12345 ...)"
        sql_estimate = f"""
            EXPLAIN SELECT 1 FROM agdc.dataset
            WHERE (metadata->'properties'->>'datetime')::timestamp >= '{year}-01-01'
            AND   (metadata->'properties'->>'datetime')::timestamp <= '{year}-12-31 23:59:59'
        """
        cur.execute(sql_estimate)
        explain_result = cur.fetchone()[0] # Lấy dòng đầu tiên của kết quả EXPLAIN
        
        # Dùng Regex để bắt lấy con số sau chữ "rows="
        match = re.search(r"rows=(\d+)", explain_result)
        if match:
            count = int(match.group(1))
        else:
            count = 0 # Không bắt được số
            
        total_est_rows += count
        est_size_mb = (count * AVG_ROW_SIZE_KB) / 1024
        
        print(f"{year:<10} | ~ {count:<18,} | ~ {est_size_mb:.2f} MB")

    print("-" * 60)
    print(f"TỔNG CỘNG : ~ {total_est_rows:,} ảnh (Khoảng {(total_est_rows * AVG_ROW_SIZE_KB)/1024:.2f} MB)")
    print("-" * 60)
    print("(Lưu ý: Số liệu này là ước tính của Database, độ chính xác khoảng 80-90%)")

    # ==========================================
    # QUYẾT ĐỊNH TẢI
    # ==========================================
    check = input("\nBạn có muốn bắt đầu tải không? (y/n): ")
    if check.lower() != 'y':
        print("Đã hủy.")
        exit()

    # ==========================================
    # BƯỚC 2 & 3: TẢI DỮ LIỆU (Giữ nguyên logic cũ)
    # ==========================================
    # ... (Phần code tải small_tables và tải dữ liệu chính giữ nguyên như cũ) ...
    # Để code gọn, mình viết tiếp phần tải ở dưới đây:
    
    # 2. Tải bảng nhỏ
    print("\n>>> Đang tải bảng định nghĩa...")
    for tb in ['agdc.metadata_type', 'agdc.dataset_type']:
        f_name = tb.split('.')[1] + ".csv"
        with open(f_name, 'w') as f:
            cur.copy_expert(f"COPY (SELECT * FROM {tb}) TO STDOUT WITH CSV HEADER", f)
            
    # 3. Tải dữ liệu chính
    print("\n>>> Đang tải dữ liệu chính...")
    for year in YEARS:
        print(f"--> Năm {year}...")
        
        # Dataset
        f_ds = f"dataset_{year}.csv"
        sql_ds = f"""
        COPY (SELECT * FROM agdc.dataset 
              WHERE (metadata->'properties'->>'datetime')::timestamp >= '{year}-01-01' 
              AND (metadata->'properties'->>'datetime')::timestamp <= '{year}-12-31 23:59:59'
        ) TO STDOUT WITH CSV HEADER"""
        with open(f_ds, 'w') as f: cur.copy_expert(sql_ds, f)
        
        # Location
        f_loc = f"dataset_location_{year}.csv"
        sql_loc = f"""
        COPY (SELECT l.* FROM agdc.dataset_location l JOIN agdc.dataset d ON l.dataset_ref = d.id
              WHERE (d.metadata->'properties'->>'datetime')::timestamp >= '{year}-01-01' 
              AND (d.metadata->'properties'->>'datetime')::timestamp <= '{year}-12-31 23:59:59'
        ) TO STDOUT WITH CSV HEADER"""
        with open(f_loc, 'w') as f: cur.copy_expert(sql_loc, f)
        
    print("\nHOÀN TẤT TOÀN BỘ!")
    cur.close()
    conn.close()

except Exception as e:
    print(f"\nCÓ LỖI: {e}")

Đang kết nối...
Kết nối thành công! Đang ước lượng nhanh...

--- BẢNG DỰ TÍNH DUNG LƯỢNG (ƯỚC LƯỢNG) ---
NĂM        | SỐ ẢNH (ƯỚC TÍNH)    | SIZE (CSV)          
------------------------------------------------------------
2020       | ~ 21,448             | ~ 83.78 MB
2021       | ~ 21,448             | ~ 83.78 MB
2022       | ~ 21,448             | ~ 83.78 MB
2023       | ~ 21,448             | ~ 83.78 MB
2024       | ~ 21,448             | ~ 83.78 MB
------------------------------------------------------------
TỔNG CỘNG : ~ 107,240 ảnh (Khoảng 418.91 MB)
------------------------------------------------------------
(Lưu ý: Số liệu này là ước tính của Database, độ chính xác khoảng 80-90%)



Bạn có muốn bắt đầu tải không? (y/n):  y



>>> Đang tải bảng định nghĩa...

>>> Đang tải dữ liệu chính...
--> Năm 2020...


In [2]:
import psycopg2
import os
import pandas as pd

# CẤU HÌNH: Ước lượng dung lượng trung bình cho mỗi cảnh ảnh (Đơn vị: GB)
# Đây là con số kinh nghiệm thực tế:
AVG_SIZE_GB = {
    's2_l2a': 0.8,       # Sentinel-2 L2A (Khoảng 800MB/cảnh)
    'ga_s2_gm': 0.1,     # Geomedian (Ảnh tổng hợp, nhẹ hơn)
    'ls5_sr': 0.5,       # Landsat 5 (Nhẹ hơn S2)
    'ls7_sr': 0.6,       # Landsat 7
    'ls8_sr': 1.0,       # Landsat 8 (Nặng hơn)
    'ls9_sr': 1.0,       # Landsat 9
    'wofs_albers': 0.05, # Water Observation (Chỉ là mask nước, rất nhẹ)
    'default': 0.5       # Mặc định nếu không biết loại nào
}

# Kết nối Database
db_host = os.environ.get('DB_HOSTNAME')
db_user = os.environ.get('DB_USERNAME')
db_pass = os.environ.get('DB_PASSWORD')
db_name = os.environ.get('DB_DATABASE')

print("Đang quét kho dữ liệu (Điều này có thể mất 1-2 phút)...")

try:
    conn = psycopg2.connect(
        host=db_host, user=db_user, password=db_pass, dbname=db_name
    )
    
    # Query: Đếm số lượng ảnh theo từng loại Product
    sql = """
    SELECT t.name as product_name, count(*) as total_images
    FROM agdc.dataset d
    JOIN agdc.dataset_type t ON d.dataset_type_ref = t.id
    GROUP BY t.name
    ORDER BY total_images DESC;
    """
    
    df = pd.read_sql(sql, conn)
    
    # Tính toán dung lượng
    def estimate_size(row):
        # Tìm size trong từ điển, nếu không có thì lấy default
        size_per_scene = AVG_SIZE_GB.get(row['product_name'], AVG_SIZE_GB['default'])
        return row['total_images'] * size_per_scene

    df['total_size_gb'] = df.apply(estimate_size, axis=1)
    df['total_size_tb'] = df['total_size_gb'] / 1024
    
    # Hiển thị báo cáo
    print("\n" + "="*60)
    print(f"{'PRODUCT NAME':<25} | {'SỐ LƯỢNG':<10} | {'DUNG LƯỢNG (TB)':<15}")
    print("-" * 60)
    
    for index, row in df.iterrows():
        print(f"{row['product_name']:<25} | {row['total_images']:<10,} | {row['total_size_tb']:.2f} TB")
        
    print("-" * 60)
    total_tb = df['total_size_tb'].sum()
    print(f"TỔNG CỘNG TOÀN SERVER  : ~ {total_tb:.2f} TB (Terabytes)")
    print(f"Tương đương            : ~ {total_tb * 1024:.0f} GB")
    print("="*60)

    conn.close()

except Exception as e:
    print(f"Lỗi: {e}")

Đang quét kho dữ liệu (Điều này có thể mất 1-2 phút)...


/tmp/ipykernel_499/84613258.py:40: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)



PRODUCT NAME              | SỐ LƯỢNG   | DUNG LƯỢNG (TB)
------------------------------------------------------------
s2_l2a                    | 2,074,896  | 1621.01 TB
sentinel_2_c1_l2a         | 1,137,446  | 555.39 TB
landsat7_c2l2_sr          | 169,065    | 82.55 TB
landsat7_c2l2_st          | 168,410    | 82.23 TB
landsat8_c2l2_sr          | 122,846    | 59.98 TB
landsat8_c2l2_st          | 120,778    | 58.97 TB
landsat5_c2l2_sr          | 118,583    | 57.90 TB
landsat5_c2l2_st          | 118,406    | 57.82 TB
landsat8_c2l1             | 111,513    | 54.45 TB
nasa_aqua_l2_oc           | 37,232     | 18.18 TB
landsat9_c2l2_sr          | 30,214     | 14.75 TB
landsat9_c2l2_st          | 29,709     | 14.51 TB
landsat9_c2l1             | 18,875     | 9.22 TB
nasa_aqua_l2_sst          | 12,070     | 5.89 TB
ga_ls_mangrove_cover_cyear_3 | 5,580      | 2.72 TB
sentinel1_grd_gamma0_10m_unsmooth | 5,488      | 2.68 TB
sentinel1_grd_gamma0_20m  | 4,260      | 2.08 TB
copernicus_dem_30     